# MLOps Architecture: Heart Disease Prediction

## Project overview

This notebook presents the end-to-end architecture of the heart disease prediction MLOps project. It describes how source data progresses through preprocessing and model development into a containerized inference service, and how automation and monitoring support reliable operation.

The design distinguishes **model development and tracking**, **continuous integration and image building**, and **deployment, inference, and monitoring**. This separation makes the workflow easier to understand, validate, and extend.

## 1. Architecture flow

### Model development and packaging
1. The **UCI Heart Disease dataset** provides the source records for model development.
2. **Data preprocessing** validates and transforms the data into features and labels suitable for training.
3. **Model training and evaluation** fit candidate classifiers and compare their performance.
4. **MLflow** records experiment runs, parameters, metrics, and associated artifacts for traceability.
5. The selected model and its matching preprocessor are saved as **model artifacts** for use during inference.

### CI/CD and deployment
6. **GitHub Actions** automates dependency installation, Flake8 linting, pytest, model training, artifact checks, and Docker image building.
7. The **Docker image** packages the FastAPI application and the model artifacts required to serve predictions.
8. **Kubernetes** runs the containerized API as a deployment. Making the image available to the cluster and applying the deployment are separate from the current CI workflow.

### Inference and monitoring
9. A client sends a prediction request to the **FastAPI inference API** and receives a prediction response.
10. The API exposes operational metrics at **`/metrics`**. A Prometheus server can scrape this endpoint for monitoring.

## 2. Architecture diagram

The diagram separates offline model-development activities from CI and runtime operations. A dashed connection indicates that image publication/deployment or metric scraping is an integration step rather than an automatic action performed by the adjacent component.

```mermaid
flowchart LR
    subgraph Development["Model Development"]
        UCI["UCI Heart Disease Dataset"]
        Prep["Data Preprocessing"]
        Train["Model Training and Evaluation"]
        MLflow[("MLflow Experiment Tracking")]
        Artifacts[("Model Artifact Storage<br/>Model + Preprocessor")]

        UCI --> Prep --> Train
        Train <--> |"Parameters, metrics,<br/>runs and artifacts"| MLflow
        Train --> |"Selected model<br/>and preprocessor"| Artifacts
    end

    subgraph CICD["CI/CD — GitHub Actions"]
        Repo["GitHub Repository"]
        Checks["Install Dependencies<br/>Flake8 + Pytest"]
        CITrain["Train Model<br/>Verify Artifacts"]
        Build["Build Docker Image"]

        Repo --> Checks --> CITrain --> Build
    end

    Artifacts -. "Model artifacts included<br/>in application image" .-> Build

    subgraph Runtime["Serving and Monitoring"]
        Image["Docker Image<br/>FastAPI + model artifacts"]
        K8s["Kubernetes Deployment<br/>API replicas"]
        API["FastAPI Inference API"]
        Client["Prediction Client"]
        Prometheus["Prometheus"]
        Metrics["/metrics Endpoint"]

        Image --> K8s --> API
        Client --> |"Prediction request"| API
        API --> |"Prediction response"| Client
        API --> Metrics
        Prometheus -. "Scrapes metrics" .-> Metrics
    end

    Build -. "Image made available to cluster<br/>and deployed separately" .-> Image
```

## 3. Component responsibilities

| Component | Responsibility |
|---|---|
| UCI Heart Disease dataset | Supplies the source data used to develop and evaluate the prediction model. |
| Data preprocessing | Validates records and prepares consistent model inputs, including the transformations needed at inference time. |
| Model training and evaluation | Fits candidate models and selects a model based on measured performance. |
| MLflow | Tracks experiment runs, parameters, metrics, and artifacts to support reproducibility and comparison. |
| Model artifact storage | Holds the selected model and corresponding preprocessor used by the API. |
| FastAPI | Provides HTTP endpoints for health checks and model inference, and exposes metrics at `/metrics`. |
| Docker | Packages the API and its runtime dependencies into a portable container image. |
| Kubernetes | Runs and manages API containers as a deployment, supporting replicated serving and health-based lifecycle management. |
| Prometheus | Scrapes the API metrics endpoint to collect operational measurements for monitoring. |
| GitHub Actions | Automates code quality checks, tests, model training/artifact validation, and Docker image building. |

## 4. Explanation for a university report

The project architecture demonstrates an end-to-end MLOps workflow by connecting data preparation, experiment tracking, model packaging, automated validation, deployment, and monitoring. The workflow begins with the UCI Heart Disease dataset. Preprocessing prepares the data for model training and ensures that the same feature transformations can be applied when serving predictions. Candidate models are trained and evaluated, while MLflow records experiment details to improve traceability and reproducibility. The chosen model and its matching preprocessor are retained as artifacts for inference.

GitHub Actions provides an automated quality gate for repository changes. The current workflow installs dependencies, runs Flake8 and pytest, trains a model, verifies that required artifacts exist, and builds a Docker image. Training occurs before image construction so that the required model and preprocessor artifacts can be included in the image. **The current workflow builds the image but does not publish it to a container registry or automatically deploy it to Kubernetes.** Those steps must be configured separately if fully automated deployment is required.

At runtime, the Docker image runs the FastAPI inference service in Kubernetes. Clients communicate with the API to request predictions, while Kubernetes manages the application deployment. The `/metrics` endpoint exposes API metrics in a format that a Prometheus server can scrape. This provides operational observability for the serving layer; MLflow, in contrast, records information about model experiments. Together, these components support a reproducible development process and a deployable, observable prediction service.

## 5. Summary

This architecture links the UCI dataset to a tracked and packaged machine-learning model, then serves that model through FastAPI in a Docker container managed by Kubernetes. GitHub Actions automates testing, linting, model artifact validation, and image building, while Prometheus-compatible metrics provide a basis for monitoring API behavior. The separation between experiment tracking, CI validation, deployment, and runtime monitoring clarifies each component's role and provides a foundation for future improvements such as image publication and automated Kubernetes deployment.